# 15. Convolutional Neural Network (CNN)
**Use case:** Oil & Gas — Pipeline inspection - detecting corrosion pits and cracks in image patches

Pipeline operators collect huge volumes of in-line inspection and camera data, and analysts must find small corrosion pits and cracks before they grow into leaks or ruptures. Manual review is slow, tiring and inconsistent across analysts, while a missed defect can harm people, the environment and the operator's licence to operate. The data is grayscale image patches of pipe wall, a small share of which contain defects confirmed by digs. Success means catching nearly every real defect, keeping false calls low enough that analysts are not flooded, and showing analysts which pixels triggered each flag.

> 💡 An inspector runs a small magnifying glass across a pipe photo, square by square, looking for tell-tale shapes - a dark dot, a thin line. A CNN does the same with many 'magnifying glasses' at once, and learns from examples which shapes matter.

Legend in code: `# 🐍 PY:` Python lesson · `# 🧠 ML:` ML lesson

In [ ]:
# Run once (Colab already has most of these)
!pip -q install numpy pandas scikit-learn scipy xgboost torch

## CONCEPT: One 3x3 filter finds a crack by hand
Slide a hand-made 3x3 crack filter over a 6x6 patch in numpy and see where it lights up.

| 5W + H | |
|---|---|
| **Who** | An integrity engineer curious what a CNN actually computes. |
| **What** | A 6x6 grayscale patch with one dark vertical crack and a clean patch for comparison. |
| **When** | Before training any network, to understand convolution itself. |
| **Where** | A short numpy script. |
| **Why** | Every CNN layer is just many of these filters, learned instead of hand-made. |
| **How** | Nested loops slide the filter, multiply and sum, then apply ReLU and max pooling to decide. |

**🧠 ML concepts:** Convolution as sliding multiply-and-sum, Filters as pattern detectors, Feature maps, ReLU activation, Max pooling to a single score  
**🐍 Python concepts:** 2-D numpy arrays, Tuple unpacking of .shape, Nested for loops, 2-D slicing, Element-wise multiply with *, Generator expression inside max

In [ ]:
import numpy as np

# A tiny 6x6 grayscale patch of pipe wall: 9 = bright clean steel, 1 = dark crack line
# 🐍 PY: a list of lists becomes a 2-D numpy array (rows x columns)
patch = np.array([
    [9, 9, 9, 1, 9, 9],
    [9, 9, 9, 1, 9, 9],
    [9, 9, 9, 1, 9, 9],
    [9, 9, 9, 1, 9, 9],
    [9, 9, 9, 1, 9, 9],
    [9, 9, 9, 1, 9, 9],
], dtype=float)
print("Inspection patch (dark column = crack):")
print(patch.astype(int))

# 🧠 ML: a 3x3 filter (kernel) - this one responds to dark vertical lines between bright pixels
# 🧠 ML: bright neighbours (+1) minus twice the centre (-2): a DARK centre column gives a BIG positive value
kernel = np.array([
    [1, -2, 1],
    [1, -2, 1],
    [1, -2, 1],
], dtype=float)
print("\nFilter (vertical dark-line detector):")
print(kernel.astype(int))

# 🧠 ML: convolution = slide the filter over the image, multiply overlapping numbers, add them up
# 🐍 PY: shape unpacking gives height and width in one line
h, w = patch.shape
out = np.zeros((h - 2, w - 2))
# 🐍 PY: nested loops visit every position where the 3x3 window fits
for i in range(h - 2):
    for j in range(w - 2):
        # 🐍 PY: slicing [i:i+3, j:j+3] cuts out the 3x3 window under the filter
        window = patch[i:i + 3, j:j + 3]
        # 🐍 PY: * is element-wise multiply for numpy arrays; .sum() adds all 9 products
        out[i, j] = (window * kernel).sum()

print("\nFeature map after convolution:")
print(out.astype(int))

# 🧠 ML: ReLU keeps positive evidence and zeroes the rest - same as in a real CNN layer
relu = np.maximum(out, 0)
# 🧠 ML: max pooling = 'is the pattern present anywhere?' -> one strong number
score = relu.max()
# 🐍 PY: np.unravel_index turns a flat position into a (row, col) pair
row, col = np.unravel_index(relu.argmax(), relu.shape)
print(f"\nStrongest crack response: {score:.0f} at feature-map column {col} (image column {col + 1})")
print("Verdict:", "DEFECT - send for dig verification" if score > 20 else "clean")

# Same filter on a clean patch with no crack
clean = np.full((6, 6), 9.0)
clean_score = max((clean[i:i + 3, j:j + 3] * kernel).sum() for i in range(4) for j in range(4))
print(f"Clean patch response: {clean_score:.0f} -> clean")

**Try this next:**
- Make the crack horizontal (a dark row) and see the vertical filter miss it; then transpose the kernel.
- Change the crack pixels from 1 to 6 (faint crack) and see whether the score still passes 20.
- Add random noise with np.random.normal(0, 1, (6, 6)) to the clean patch and check for false alarms.

## BUILD: Tiny PyTorch CNN on 1,200 inspection patches
Generate synthetic pipe-wall patches with pits and cracks and train a two-layer PyTorch CNN to flag defects.

| 5W + H | |
|---|---|
| **Who** | A data scientist supporting the pipeline integrity team. |
| **What** | 1,200 synthetic 16x16 grayscale patches, about 30% with a pit or crack. |
| **When** | When moving from hand-made filters to filters learned from examples. |
| **Where** | A CPU Python job - the model has only about 1,500 weights. |
| **Why** | Learned filters handle defects of any size, angle and position far better than hand rules. |
| **How** | Two Conv-ReLU-MaxPool blocks and a linear output trained with BCE loss and Adam for 10 epochs. |

**🧠 ML concepts:** Learned convolution filters, Stacked conv and pooling layers, Channels and tensor shapes, Binary cross-entropy with logits, Recall as the key metric for safety  
**🐍 Python concepts:** Functions with branches, np.mgrid coordinate grids, List comprehension plus np.stack, unsqueeze to add a dimension, Generator expression in sum, Boolean masks combined with &

In [ ]:
import numpy as np
import torch
import torch.nn as nn

torch.set_num_threads(1)
torch.manual_seed(42)
rng = np.random.default_rng(42)
SIZE = 16  # 16x16 pixel inspection patches


# --- 1. Generate synthetic pipe-wall inspection patches ---
def make_patch(defect):
    # 🧠 ML: normal steel = bright, slightly grainy texture with smooth shading
    # 🐍 PY: np.linspace + broadcasting builds a gentle left-to-right brightness gradient
    img = 0.7 + 0.1 * np.linspace(-1, 1, SIZE)[None, :] + rng.normal(0, 0.05, (SIZE, SIZE))
    if defect:
        if rng.random() < 0.5:
            # Corrosion pit: a dark round blob somewhere in the patch
            cy, cx, r = rng.integers(3, SIZE - 3, 2).tolist() + [rng.uniform(1.5, 3)]
            yy, xx = np.mgrid[:SIZE, :SIZE]
            img -= 0.35 * np.exp(-((yy - cy) ** 2 + (xx - cx) ** 2) / (2 * r ** 2))
        else:
            # Crack: a thin dark line at a random angle
            angle, cx, cy = rng.uniform(0, np.pi), rng.uniform(5, 11), rng.uniform(5, 11)
            for t in np.linspace(-6, 6, 40):
                y, x = int(cy + t * np.sin(angle)), int(cx + t * np.cos(angle))
                if 0 <= y < SIZE and 0 <= x < SIZE:
                    img[y, x] -= 0.3
    return np.clip(img, 0, 1)

n = 1200
# 🐍 PY: rng.random(n) < 0.3 gives a True/False array -> about 30% defects
labels = (rng.random(n) < 0.3).astype(int)
# 🐍 PY: list comprehension builds each patch; np.stack turns the list into one (n, 16, 16) array
images = np.stack([make_patch(d) for d in labels])
print(f"Patches: {images.shape}, defect rate {labels.mean():.1%}")

# 🧠 ML: CNNs expect (batch, channels, height, width) - grayscale has 1 channel
X = torch.tensor(images, dtype=torch.float32).unsqueeze(1)
y = torch.tensor(labels, dtype=torch.float32)
split = int(0.8 * n)
X_tr, X_te, y_tr, y_te = X[:split], X[split:], y[:split], y[split:]

# --- 2. Tiny CNN ---
# 🧠 ML: Conv2d learns its own 3x3 filters (like the hand-made crack filter in the easy level)
# 🧠 ML: MaxPool halves the image size and keeps the strongest response in each 2x2 block
model = nn.Sequential(
    nn.Conv2d(1, 8, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # 16x16 -> 8x8
    nn.Conv2d(8, 16, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 8x8 -> 4x4
    nn.Flatten(),
    nn.Linear(16 * 4 * 4, 1),
)
# 🐍 PY: sum over a generator expression counts every trainable number in the model
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters())}")
opt = torch.optim.Adam(model.parameters(), lr=0.003)
loss_fn = nn.BCEWithLogitsLoss()

# --- 3. Train with mini-batches ---
for epoch in range(1, 11):
    perm = torch.randperm(split)
    # 🐍 PY: .split(64) cuts the shuffled index list into chunks of 64
    for idx in perm.split(64):
        opt.zero_grad()
        loss = loss_fn(model(X_tr[idx]).squeeze(1), y_tr[idx])
        loss.backward()
        opt.step()
    if epoch % 2 == 0:
        print(f"epoch {epoch:2d}  last batch loss {loss.item():.3f}")

# --- 4. Evaluate ---
with torch.no_grad():
    prob = torch.sigmoid(model(X_te).squeeze(1)).numpy()
pred = (prob >= 0.5).astype(int)
truth = y_te.numpy().astype(int)
# 🐍 PY: boolean arithmetic - (pred == 1) & (truth == 1) is True only where both hold
tp = int(((pred == 1) & (truth == 1)).sum())
fp = int(((pred == 1) & (truth == 0)).sum())
fn = int(((pred == 0) & (truth == 1)).sum())
print(f"\nTest accuracy: {(pred == truth).mean():.3f}")
print(f"Defects caught: {tp}/{tp + fn}  (recall {tp / (tp + fn):.2f})")
print(f"False calls on clean steel: {fp}  (precision {tp / max(tp + fp, 1):.2f})")
print("\nInspection read-out: patches flagged as defects go to an integrity engineer for review;")
print("missed defects (false negatives) are the costly error, so recall is the key number.")

**Try this next:**
- Train for 20 epochs instead of 10 and see whether recall improves.
- Lower the decision threshold from 0.5 to 0.3 and compare recall and false calls.
- Change the first Conv2d to 4 filters and then 16 filters and compare results.

## PRODUCTION: Recall-first CNN with augmentation and saliency
Train a production-style defect CNN with augmentation and class weighting, set a recall-first threshold on validation data, compare with a pixel baseline and explain flags with saliency.

| 5W + H | |
|---|---|
| **Who** | The integrity data-science team, with integrity analysts and regulatory compliance as users. |
| **What** | A saved CNN and threshold that flags pits and cracks, with per-defect-type recall and a pixel-level explanation. |
| **When** | After each in-line inspection run; revalidated whenever field digs reveal misses. |
| **Where** | A batch scoring service feeding flagged patches to the analyst review tool. |
| **Why** | Missed defects can lead to failures, so the model must hit a recall target, and analysts need to see why it flagged a patch. |
| **How** | Nuisance weld seams in data, flips and rotations as augmentation, batch norm, global max pooling, pos_weight for imbalance, best-validation weights, a threshold chosen for 95% validation recall, a logistic-regression baseline and gradient saliency. |

**🧠 ML concepts:** Data augmentation for orientation invariance, Batch normalisation, Global max pooling for position independence, Class weighting for rare defects, Recall-first threshold on validation data, Gradient saliency for explanation  
**🐍 Python concepts:** Classes with configurable __init__, slice objects for reusable index ranges, List comprehension with a condition, Unpacking tuples from a list, String indexing to draw ASCII art, zip to print two maps side by side

In [ ]:
"""Pipeline defect detection CNN: augmentation, train/val/test, recall-first threshold,
confusion matrix, baseline comparison, gradient saliency map, model saving, PHMSA note."""
import os
import tempfile

import numpy as np
import torch
import torch.nn as nn
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, precision_score, recall_score

torch.set_num_threads(1)
torch.manual_seed(42)
rng = np.random.default_rng(42)
SIZE = 16


def make_patch(kind):
    """kind: 0 = clean, 1 = corrosion pit, 2 = crack. Adds weld seams and grain as nuisances."""
    img = 0.7 + 0.1 * np.linspace(-1, 1, SIZE)[None, :] + rng.normal(0, 0.05, (SIZE, SIZE))
    # 🧠 ML: nuisance features (a bright weld seam) on ANY patch stop the model using 'any edge = defect'
    if rng.random() < 0.3:
        img[rng.integers(2, SIZE - 2), :] += 0.15
    if kind == 1:
        cy, cx = rng.integers(3, SIZE - 3, 2)
        r = rng.uniform(1.2, 2.5)
        yy, xx = np.mgrid[:SIZE, :SIZE]
        img -= rng.uniform(0.1, 0.3) * np.exp(-((yy - cy) ** 2 + (xx - cx) ** 2) / (2 * r ** 2))
    elif kind == 2:
        angle, cx, cy = rng.uniform(0, np.pi), rng.uniform(5, 11), rng.uniform(5, 11)
        for t in np.linspace(-5, 5, 30):
            y, x = int(cy + t * np.sin(angle)), int(cx + t * np.cos(angle))
            if 0 <= y < SIZE and 0 <= x < SIZE:
                img[y, x] -= rng.uniform(0.08, 0.25)
    return np.clip(img, 0, 1)


def make_dataset(n, defect_rate=0.15):
    kinds = np.where(rng.random(n) < defect_rate, rng.integers(1, 3, n), 0)
    X = np.stack([make_patch(k) for k in kinds]).astype(np.float32)
    return X, (kinds > 0).astype(np.float32), kinds


def augment(xb):
    # 🧠 ML: defects can appear at any orientation - random flips/rotations teach that for free
    if rng.random() < 0.5:
        xb = torch.flip(xb, dims=[3])
    if rng.random() < 0.5:
        xb = torch.flip(xb, dims=[2])
    return torch.rot90(xb, k=int(rng.integers(0, 4)), dims=[2, 3])


class DefectCNN(nn.Module):
    def __init__(self, c1=8, c2=16):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, c1, 3, padding=1), nn.BatchNorm2d(c1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(c1, c2, 3, padding=1), nn.BatchNorm2d(c2), nn.ReLU(), nn.MaxPool2d(2))
        # 🧠 ML: global max pooling asks 'is a defect pattern present ANYWHERE?' - position-independent
        self.head = nn.Sequential(nn.AdaptiveMaxPool2d(1), nn.Flatten(), nn.Linear(c2, 1))

    def forward(self, x):
        return self.head(self.features(x)).squeeze(1)


def train(model, Xtr, ytr, Xval, yval, epochs=15, use_aug=True):
    # 🧠 ML: pos_weight counters the 15% defect rate so missed defects cost more during training
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=(ytr == 0).sum() / (ytr == 1).sum())
    opt = torch.optim.Adam(model.parameters(), lr=3e-3)
    best, best_state = np.inf, None
    for _ in range(epochs):
        model.train()
        for idx in torch.randperm(len(Xtr)).split(64):
            xb = augment(Xtr[idx]) if use_aug else Xtr[idx]
            opt.zero_grad()
            loss_fn(model(xb), ytr[idx]).backward()
            opt.step()
        model.eval()
        with torch.no_grad():
            v = loss_fn(model(Xval), yval).item()
        if v < best:
            best, best_state = v, {k: t.clone() for k, t in model.state_dict().items()}
    model.load_state_dict(best_state)
    return model


def predict(model, X):
    model.eval()
    with torch.no_grad():
        return torch.sigmoid(model(X)).numpy()


def saliency(model, x):
    """Gradient of the defect score w.r.t. each pixel: which pixels would change the decision most."""
    model.eval()
    x = x.clone().requires_grad_(True)
    model(x).sum().backward()
    return x.grad.abs().squeeze().numpy()


def ascii_map(a):
    # 🐍 PY: index into a string of 'ink levels' to draw a heatmap as text
    shades = " .:-=+*#"
    a = (a - a.min()) / (a.max() - a.min() + 1e-9)
    return ["".join(shades[int(v * 7)] for v in row) for row in a]


if __name__ == "__main__":
    X, y, kinds = make_dataset(2400)
    Xt, yt = torch.tensor(X).unsqueeze(1), torch.tensor(y)
    tr, va, te = slice(0, 1600), slice(1600, 2000), slice(2000, 2400)
    print(f"Train/val/test: 1600/400/400   defect rate {y.mean():.1%}")

    # 🧠 ML: baseline - logistic regression on raw pixels (no notion of local shape)
    lr = LogisticRegression(max_iter=2000, class_weight="balanced").fit(X[tr].reshape(1600, -1), y[tr])
    p_lr = lr.predict_proba(X[te].reshape(400, -1))[:, 1]

    results = {}
    for name, aug in [("CNN no-aug", False), ("CNN + aug", True)]:
        torch.manual_seed(42)
        m = train(DefectCNN(), Xt[tr], yt[tr], Xt[va], yt[va], use_aug=aug)
        results[name] = (m, predict(m, Xt[va]), predict(m, Xt[te]))

    # 🧠 ML: integrity work is recall-first - choose the HIGHEST threshold on VAL that keeps recall >= 95%
    print("\nModel        threshold  test recall  test precision  pits caught  cracks caught")
    rows = [("LogReg pix", None, 0.5, p_lr)]
    for name, (m, p_val, p_te) in results.items():
        cands = [t for t in np.arange(0.05, 0.96, 0.05) if recall_score(y[va], p_val >= t) >= 0.95]
        rows.append((name, m, max(cands) if cands else 0.5, p_te))
    for name, _, t, p in rows:
        pred = p >= t
        pits = pred[kinds[te] == 1].mean()
        cracks = pred[kinds[te] == 2].mean()
        print(f"{name:<12} {t:9.2f}  {recall_score(y[te], pred):11.2f}  "
              f"{precision_score(y[te], pred):14.2f}  {pits:11.0%}  {cracks:13.0%}")

    name, model, thr, p_te = rows[-1]
    cm = confusion_matrix(y[te], p_te >= thr)
    print(f"\nConfusion matrix for {name} (rows = truth clean/defect, cols = predicted):")
    print(f"               pred clean  pred defect\n  true clean   {cm[0, 0]:10d}  {cm[0, 1]:11d}"
          f"\n  true defect  {cm[1, 0]:10d}  {cm[1, 1]:11d}")

    # 🧠 ML: saliency on the most confident corrosion pit - bright marks should sit on the pit itself
    i = int(np.argmax(np.where(kinds[te] == 1, p_te, -1)))
    sal = saliency(model, Xt[te][i:i + 1])
    img_rows, sal_rows = ascii_map(-X[te][i]), ascii_map(sal)
    print(f"\nTest patch {i} ({['clean', 'pit', 'crack'][kinds[te][i]]}, p={p_te[i]:.2f})"
          f"   darkness map  |  saliency map")
    for a, b in zip(img_rows, sal_rows):
        print(f"   {a}  |  {b}")

    path = os.path.join(tempfile.mkdtemp(), "defect_cnn.pt")
    torch.save({"state_dict": model.state_dict(), "threshold": float(thr)}, path)
    print(f"\nSaved model and threshold -> {os.path.basename(path)}")
    print("PHMSA note: 49 CFR 192/195 integrity management - model flags support, not replace,")
    print("qualified ILI analysis; log every miss found in the field and revalidate the threshold.")

**Try this next:**
- Raise the recall target from 0.95 to 0.99 and see how many extra false calls appear.
- Remove the weld-seam line in make_patch and check whether the baseline gets much better.
- Shrink the defect depth further (0.05 to 0.15) and compare augmentation vs no augmentation.